# SFT for Qwen3.5 0.8B with Unsloth + TRL

This notebook fine-tunes a Qwen 0.8B model on balanced Energy Grid SFT data with validation + early stopping to reduce overfitting.

In [ ]:
!pip install -q --upgrade unsloth trl transformers datasets accelerate peft bitsandbytes

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from datasets import load_dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments, EarlyStoppingCallback

MODEL_NAME = "Qwen/Qwen3-0.8B"
# 512 is enough for these prompts (~300-400 tokens each) and halves VRAM vs 1024.
MAX_SEQ_LENGTH = 512
DTYPE = None
LOAD_IN_4BIT = True

TRAIN_FILE = "grid_expert_sft_train.jsonl"
VAL_FILE   = "grid_expert_sft_val.jsonl"

OUTPUT_DIR = "outputs/qwen35_08b_energygrid_sft"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
import random as _random

# Feed as many samples as possible — max_steps will enforce the 1-hour limit.
MAX_TRAIN_SAMPLES = 15_000
MAX_VAL_SAMPLES   = 500

raw = load_dataset(
    "json",
    data_files={"train": TRAIN_FILE, "validation": VAL_FILE},
)

def to_text(example):
    return {"text": f"### Prompt:\n{example['prompt']}\n\n{example['completion']}"}

original_cols = raw["train"].column_names

# Stratified subsample: shuffle then take first N so class mix is preserved.
_rng = _random.Random(42)
train_indices = list(range(len(raw["train"])))
_rng.shuffle(train_indices)
train_subset = raw["train"].select(train_indices[:MAX_TRAIN_SAMPLES])

val_indices = list(range(len(raw["validation"])))
_rng.shuffle(val_indices)
val_subset = raw["validation"].select(val_indices[:MAX_VAL_SAMPLES])

dataset = {
    "train":      train_subset.map(to_text, remove_columns=original_cols),
    "validation": val_subset.map(to_text, remove_columns=original_cols),
}

print(f"Using {len(dataset['train'])} train / {len(dataset['validation'])} val samples")

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=8,           # halved from 16 — saves ~200 MB VRAM, still enough for SFT
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

In [ ]:
import math

# ── Auto-scale hyperparameters to (capped) dataset size ─────────────────────
n_train = len(dataset["train"])
n_val   = len(dataset["validation"])
print(f"Dataset: {n_train} train / {n_val} val samples")

PER_DEVICE_BS  = 2    # low to avoid OOM on T4 (15 GB)
GRAD_ACCUM     = 8    # effective BS stays 16
EFFECTIVE_BS   = PER_DEVICE_BS * GRAD_ACCUM

# Hard time ceiling: ~3 s/step on T4, so 1 200 steps ≈ 1 hour.
# Raise to 1 800 on an A100 (≈ 1 s/step) if you have one.
MAX_STEPS      = 1_200

steps_per_epoch = max(1, math.ceil(n_train / EFFECTIVE_BS))
warmup_steps    = max(5, int(MAX_STEPS * 0.05))
eval_steps      = 100   # evaluate every 100 steps regardless of epoch length
log_steps       = 20

print(f"Effective batch size : {EFFECTIVE_BS}")
print(f"Steps per epoch      : {steps_per_epoch}")
print(f"Hard max steps       : {MAX_STEPS}  (~{MAX_STEPS*3//60} min on T4)")
print(f"Eval / save steps    : {eval_steps}")
print(f"Warmup steps         : {warmup_steps}")

# ── TrainingArguments ────────────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=PER_DEVICE_BS,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=2e-4,
    max_steps=MAX_STEPS,          # hard ceiling — stops at ~1 hour
    warmup_steps=warmup_steps,
    lr_scheduler_type="cosine",
    fp16=False,
    bf16=True,
    logging_steps=log_steps,
    evaluation_strategy="steps",
    eval_steps=eval_steps,
    save_strategy="steps",
    save_steps=eval_steps,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    dataloader_num_workers=0,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=True,   # pack multiple short samples into one sequence → 3-5× faster
    args=training_args,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

train_result = trainer.train()
train_result

In [ ]:
import matplotlib.pyplot as plt

history = trainer.state.log_history

train_steps, train_loss = [], []
eval_steps,  eval_loss  = [], []

for entry in history:
    if "loss" in entry and "eval_loss" not in entry:
        train_steps.append(entry["step"])
        train_loss.append(entry["loss"])
    if "eval_loss" in entry:
        eval_steps.append(entry["step"])
        eval_loss.append(entry["eval_loss"])

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(train_steps, train_loss, label="Train loss",      color="steelblue", linewidth=1.5)
ax.plot(eval_steps,  eval_loss,  label="Val loss",        color="tomato",    linewidth=2, marker="o", markersize=4)
ax.set_xlabel("Optimizer step")
ax.set_ylabel("Cross-entropy loss")
ax.set_title("SFT training curve — diverging val loss = overfitting")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ── Quick overfitting diagnosis ───────────────────────────────────────────────
if len(eval_loss) >= 2:
    best_val   = min(eval_loss)
    final_val  = eval_loss[-1]
    final_train = train_loss[-1] if train_loss else float("nan")
    gap = final_val - final_train

    print(f"\nBest val loss  : {best_val:.4f}  (step {eval_steps[eval_loss.index(best_val)]})")
    print(f"Final val loss : {final_val:.4f}")
    print(f"Final train loss: {final_train:.4f}")
    print(f"Generalisation gap (val - train): {gap:+.4f}")

    if final_val > best_val * 1.05:
        print("\n⚠  Val loss rose >5% above its best — model is overfitting.")
        print("   Consider: fewer max_steps, higher lora_dropout, or smaller dataset cap.")
    elif gap > 0.5:
        print("\n⚠  Large train/val gap — possible overfitting.")
    else:
        print("\n✓  Val loss is stable — no clear overfitting.")

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Saved model + tokenizer to", OUTPUT_DIR)

In [ ]:
FastLanguageModel.for_inference(model)
sample_prompt = dataset["validation"][0]["text"]
inputs = tokenizer(sample_prompt, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=120, do_sample=False)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))